# Notebook 01: Embeddings and Semantic Similarity

Before building a RAG system, you need to understand **embeddings** — the foundation of everything.

## What is an Embedding?

An embedding is a way of representing text (or any data) as a list of numbers (a vector). The key property is that **semantically similar text produces similar vectors**.

```
"The dog ran fast"     → [0.12, -0.45, 0.89, 0.03, ...]
"The puppy sprinted"  → [0.11, -0.44, 0.91, 0.02, ...]  ← similar!
"I like pizza"        → [-0.67, 0.23, -0.12, 0.78, ...] ← very different
```

This is what enables **semantic search** — finding relevant documents by meaning, not just keyword matching.

## In this notebook you will:
1. Create embeddings using `sentence-transformers`
2. Measure similarity between sentences using cosine similarity
3. Build a simple semantic search function
4. Visualize embeddings to build intuition

## Step 1: Install and Import

In [ ]:
# Install if needed
# !pip install sentence-transformers numpy scikit-learn matplotlib

from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

print("Imports successful!")

## Step 2: Load an Embedding Model

`all-MiniLM-L6-v2` is a popular, lightweight model that produces 384-dimensional embeddings. It runs entirely locally — no API key needed.

In [ ]:
# Load the embedding model (downloads ~80MB on first run)
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Model loaded!")
print(f"Embedding dimensions: {model.get_sentence_embedding_dimension()}")

## Step 3: Create Your First Embeddings

In [ ]:
# A set of sentences with varying semantic similarity
sentences = [
    "Machine learning allows computers to learn from data",
    "AI systems can improve through experience with data",
    "Deep learning uses neural networks with many layers",
    "Neural networks are inspired by the human brain",
    "The stock market rose 2% today",
    "Investors saw gains in equities this morning",
]

# Generate embeddings — one vector per sentence
embeddings = model.encode(sentences)

print(f"Number of sentences: {len(sentences)}")
print(f"Shape of embeddings array: {embeddings.shape}")
print(f"\nFirst embedding (first 10 values): {embeddings[0][:10].round(4)}")

## Step 4: Compute Cosine Similarity

**Cosine similarity** measures the angle between two vectors:
- `1.0` = identical direction (very similar meaning)
- `0.0` = perpendicular (unrelated)
- `-1.0` = opposite direction (opposite meaning)

For text embeddings, scores typically range from 0.0 to 1.0.

In [ ]:
# Compute the full similarity matrix
sim_matrix = cosine_similarity(embeddings)

# Display as a heatmap
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine Similarity')

# Label axes with truncated sentences
short_labels = [s[:35] + '...' for s in sentences]
ax.set_xticks(range(len(sentences)))
ax.set_yticks(range(len(sentences)))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(short_labels, fontsize=8)

# Add score annotations
for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha='center', va='center', fontsize=8)

ax.set_title('Cosine Similarity Between Sentences', fontsize=13)
plt.tight_layout()
plt.show()

**Notice:** 
- Sentences 0 and 1 (both about ML) have high similarity
- Sentences 4 and 5 (both about stocks) have high similarity  
- Cross-topic pairs (ML vs stocks) have low similarity

This is the core idea behind RAG retrieval — we find the most similar chunks to the user's query.

## Step 5: Build a Simple Semantic Search Function

In [ ]:
def semantic_search(query: str, documents: list, model: SentenceTransformer, top_k: int = 3):
    """
    Find the most semantically similar documents to a query.
    
    Args:
        query: The search query
        documents: List of documents to search through
        model: The embedding model
        top_k: Number of results to return
    
    Returns:
        List of (score, document) tuples sorted by similarity
    """
    # Embed the query
    query_embedding = model.encode([query])
    
    # Embed the documents
    doc_embeddings = model.encode(documents)
    
    # Compute similarity between query and all documents
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]
    
    # Rank by score and return top_k
    ranked = sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)
    return ranked[:top_k]

In [ ]:
# A small knowledge base to search through
knowledge_base = [
    "Python is a high-level programming language known for its readability.",
    "JavaScript is primarily used for web development and runs in browsers.",
    "Machine learning models learn patterns from training data.",
    "Gradient descent is an optimization algorithm used to train neural networks.",
    "Docker containers package applications and their dependencies together.",
    "SQL is used to query and manipulate relational databases.",
    "RAG combines retrieval with language model generation for better answers.",
    "Vector databases store embeddings and enable fast similarity search.",
]

# Try different queries
queries = [
    "How do neural networks learn?",
    "What is used for web programming?",
    "How does RAG work?",
]

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 50)
    results = semantic_search(query, knowledge_base, model, top_k=2)
    for score, doc in results:
        print(f"  [{score:.3f}] {doc}")

## Step 6: Visualize Embeddings with PCA

Embeddings live in 384 dimensions — we can't visualize that directly. PCA reduces them to 2D while preserving the most important structure.

In [ ]:
# Create a richer set of sentences grouped by topic
topic_sentences = {
    "Machine Learning": [
        "Neural networks learn from data",
        "Deep learning uses many layers",
        "Training requires labeled examples",
    ],
    "Finance": [
        "Stock prices rose today",
        "Investors bought equities",
        "The market hit a new high",
    ],
    "Cooking": [
        "Preheat the oven to 350 degrees",
        "Add salt and pepper to taste",
        "Bake until golden brown",
    ],
}

all_sentences = []
all_labels = []
colors = {'Machine Learning': 'royalblue', 'Finance': 'tomato', 'Cooking': 'seagreen'}

for topic, sents in topic_sentences.items():
    all_sentences.extend(sents)
    all_labels.extend([topic] * len(sents))

# Embed all sentences
all_embeddings = model.encode(all_sentences)

# Reduce to 2D with PCA
pca = PCA(n_components=2)
reduced = pca.fit_transform(all_embeddings)

# Plot
fig, ax = plt.subplots(figsize=(9, 6))
for topic, color in colors.items():
    mask = [l == topic for l in all_labels]
    x = reduced[mask, 0]
    y = reduced[mask, 1]
    ax.scatter(x, y, c=color, label=topic, s=120, zorder=5)
    for xi, yi, sent in zip(x, y, [s for s, l in zip(all_sentences, all_labels) if l == topic]):
        ax.annotate(sent[:30], (xi, yi), textcoords='offset points', xytext=(6, 4), fontsize=7)

ax.set_title('Embedding Space (PCA 2D) — Similar topics cluster together', fontsize=12)
ax.legend()
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

print(f"\nExplained variance: {pca.explained_variance_ratio_.sum():.1%} of total variance captured in 2D")

## Key Takeaways

1. **Embeddings** convert text to vectors where semantic meaning is encoded in the geometry
2. **Cosine similarity** measures how close two vectors are (how similar two texts are in meaning)
3. **Semantic search** finds documents by meaning, not by keywords
4. Topics naturally **cluster** in embedding space — this is what makes retrieval work

---

**Next:** Notebook 02 — Instead of re-embedding documents on every search, we'll store them permanently in a **vector database** (ChromaDB).